# offline_embed.ipynb — RAG Corpus Build

**Project:** Self-Corrective RAG (CRAG) — ITMD 524, Spring 2026  
**Purpose:** Build and persist a FAISS vector index from the Apple 2024 Environmental Report QA corpus to Google Drive. The index is consumed by the baseline RAG and CRAG pipelines on Chameleon.

The encoder model is set once in Cell 1 (`ENCODER_MODEL = "all-MiniLM-L6-v2"`) and must remain identical across the offline build phase and all online query phases. Changing it requires deleting the Drive artifacts and rebuilding from scratch.

## Expected outputs

| File | Location | Approx. size |
|---|---|---|
| `corpus_index.faiss` | `MyDrive/crag-vectors/` | 1.5–2 MB |
| `id_map.json` | `MyDrive/crag-vectors/` | 800 KB–1 MB |
| `config.json` | `MyDrive/crag-vectors/` | < 1 KB |

Expected `index.ntotal`: approximately 1100–1200 sub-chunks (215 source chunks re-split at 200 tokens each).

Run all cells top-to-bottom. Do not restart the kernel between cells.

In [ ]:
# ── Cell 1: Drive Mount + All Constants ───────────────────────────────────────
import os
from google.colab import drive

# -----------------------------
# 0) Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/crag-vectors"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory  : {SAVE_DIR}")

# ── Single source of truth for all configuration ──────────────────────────────
ENCODER_MODEL        = "all-MiniLM-L6-v2"  # NEVER change without rebuilding the index
EMBEDDING_DIM        = 384                  # fixed for assertion — not inferred at runtime
MAX_TOKENS           = 256                  # encoder context window hard limit (incl. special tokens)

CHUNK_SIZE_TOKENS    = 200                  # token-based chunk size — guarantees fit within MAX_TOKENS
CHUNK_OVERLAP_TOKENS = 20                   # token overlap (~10%)

DRIVE_DIR   = SAVE_DIR
INDEX_PATH  = f"{DRIVE_DIR}/corpus_index.faiss"
IDMAP_PATH  = f"{DRIVE_DIR}/id_map.json"
CONFIG_PATH = f"{DRIVE_DIR}/config.json"

HF_DATASET_ID          = "AdamLucek/apple-environmental-report-QA-retrieval"
EXPECTED_UNIQUE_CHUNKS = 215
EMBED_BATCH_SIZE       = 64

print(f"Encoder model   : {ENCODER_MODEL}")
print(f"Chunk size      : {CHUNK_SIZE_TOKENS} tokens | Overlap: {CHUNK_OVERLAP_TOKENS} tokens | Limit: {MAX_TOKENS} tokens")

## Step 1 — Install Dependencies

`faiss-cpu` is used for index operations. The GPU-intensive work (encoding) is handled by SentenceTransformer via PyTorch, not by FAISS.

In [ ]:
!pip install -q faiss-cpu sentence-transformers datasets langchain-text-splitters transformers numpy matplotlib

In [ ]:
import faiss
import sentence_transformers
import datasets as hf_datasets
import numpy as np
import torch

print(f"faiss                 {faiss.__version__}")
print(f"sentence-transformers {sentence_transformers.__version__}")
print(f"numpy                 {np.__version__}")
print(f"torch                 {torch.__version__}")

# GPU check — SentenceTransformer uses PyTorch/CUDA for encoding (not FAISS)
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
else:
    print("\nNo GPU detected. Encoding will run on CPU.")

## Step 2 — Load Dataset and Extract Unique Chunks

The dataset has 4,300 Q&A rows across two splits, but only **215 unique text chunks**. Each chunk appears ~20 times (once per synthetic question). We deduplicate to get exactly the 215 source passages that will be indexed.

In [ ]:
# ── Cell 3: Dataset Load + Deduplication ─────────────────────────────────────
import pandas as pd
from datasets import load_dataset

print(f"Loading dataset: {HF_DATASET_ID} ...")
ds = load_dataset(HF_DATASET_ID)

train_df = pd.DataFrame(ds["train"])
val_df   = pd.DataFrame(ds["validation"])
all_df   = pd.concat([train_df, val_df], ignore_index=True)

print(f"Total rows: {len(all_df):,}  (train={len(train_df):,}, val={len(val_df):,})")

# Deduplicate on chunk text — preserve first-occurrence order
unique_chunks = all_df["chunk"].drop_duplicates().reset_index(drop=True)
chunk_list    = unique_chunks.tolist()

# Hard assertion — any deviation means the dataset changed upstream
assert len(chunk_list) == EXPECTED_UNIQUE_CHUNKS, (
    f"Expected {EXPECTED_UNIQUE_CHUNKS} unique chunks, got {len(chunk_list)}. "
    f"Dataset may have changed upstream."
)

print(f"\nUnique chunks: {len(chunk_list)}")
print(f"\nChar-length statistics across {len(chunk_list)} source chunks:")
char_lengths = pd.Series([len(c) for c in chunk_list])
print(char_lengths.describe().to_string())
print(f"\nSample chunk [0] (first 300 chars):\n{chunk_list[0][:300]}")

## Step 3 — Re-chunk to Fit the Encoder Window

Source chunks average approximately 600+ tokens, exceeding the 256-token context window of `all-MiniLM-L6-v2`. Feeding full chunks causes silent truncation, degrading retrieval quality.

The splitter uses the model's own tokenizer as the length function (`chunk_size=200 tokens, overlap=20 tokens`), guaranteeing every sub-chunk fits within the encoder limit. The `original_chunk_idx` field in each sub-chunk preserves traceability back to the 215 source passages.

In [ ]:
# ── Cell 4: Re-chunk with token-aware RecursiveCharacterTextSplitter ──────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

# Load the exact tokenizer the encoder uses — makes chunk_size mean *tokens*, not chars
tokenizer = AutoTokenizer.from_pretrained(f"sentence-transformers/{ENCODER_MODEL}")

def token_length(text: str) -> int:
    """Count tokens including [CLS] and [SEP] special tokens."""
    return len(tokenizer.encode(text, add_special_tokens=True))

# chunk_size and chunk_overlap use constants from Cell 1
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_TOKENS,
    chunk_overlap=CHUNK_OVERLAP_TOKENS,
    length_function=token_length,          # token-based, not character-based
    separators=["\n\n", "\n", ". ", " ", ""],
)

sub_chunks = []   # flat list; index position == FAISS ID

for orig_idx, source_text in enumerate(chunk_list):
    pieces = splitter.split_text(source_text)
    for piece in pieces:
        sub_chunks.append({
            "text": piece,
            "original_chunk_idx": orig_idx,
        })

# Distribution summary
from collections import Counter
counts       = Counter(sc["original_chunk_idx"] for sc in sub_chunks)
sub_per_orig = list(counts.values())

print(f"Source chunks:                {len(chunk_list)}")
print(f"Sub-chunks (will be indexed): {len(sub_chunks)}")
print(f"Avg sub-chunks per source:    {len(sub_chunks)/len(chunk_list):.1f}")
print(f"Sub-chunks per source — min: {min(sub_per_orig)}, max: {max(sub_per_orig)}, "
      f"mean: {sum(sub_per_orig)/len(sub_per_orig):.1f}")
print(f"\nSample sub-chunk [0]:\n{sub_chunks[0]['text'][:300]}")

## Step 4 — Token-Length Validation

Verifies that every sub-chunk is within the 256-token encoder limit before any embedding is computed. If this check fails, lower `CHUNK_SIZE_TOKENS` in Cell 1 and rerun from Cell 4.

In [ ]:
# ── Cell 5: Token-Length Validation ──────────────────────────────────────────
# tokenizer is already loaded in Cell 4 — reuse it here
import matplotlib.pyplot as plt

texts_only    = [sc["text"] for sc in sub_chunks]
encodings     = tokenizer(
    texts_only,
    add_special_tokens=True,
    truncation=False,
    padding=False,
)
token_lengths = np.array([len(ids) for ids in encodings["input_ids"]])
over_limit    = (token_lengths > MAX_TOKENS).sum()

print(f"Token length statistics across {len(sub_chunks)} sub-chunks:")
print(f"  min:    {token_lengths.min()}")
print(f"  median: {np.median(token_lengths):.0f}")
print(f"  mean:   {token_lengths.mean():.1f}")
print(f"  max:    {token_lengths.max()}")
print(f"  > {MAX_TOKENS} tokens: {over_limit}")

assert over_limit == 0, (
    f"{over_limit} sub-chunk(s) exceed {MAX_TOKENS} tokens (max found: {token_lengths.max()}). "
    f"Lower chunk_size in Cell 4 and rerun from Cell 4."
)
print(f"\n All {len(sub_chunks)} sub-chunks are within {MAX_TOKENS} tokens.")

plt.figure(figsize=(10, 4))
plt.hist(token_lengths, bins=40, edgecolor='black', color='steelblue', alpha=0.85)
plt.axvline(MAX_TOKENS, color='red', linestyle='--', linewidth=2,
            label=f'Hard limit ({MAX_TOKENS} tokens)')
plt.xlabel("Token count per sub-chunk (incl. special tokens)")
plt.ylabel("Count")
plt.title(f"Token Length Distribution — {len(sub_chunks)} sub-chunks")
plt.legend()
plt.tight_layout()
plt.show()

## Step 5 — Embed with `all-MiniLM-L6-v2`

In [ ]:
# ── Cell 6: Embedding ─────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

# Load encoder using the constant — NEVER hardcode the model name here
encoder    = SentenceTransformer(ENCODER_MODEL)
texts_only = [sc["text"] for sc in sub_chunks]   # re-derive to be self-contained

print(f"Encoding {len(texts_only)} sub-chunks with '{ENCODER_MODEL}'...")

embeddings = encoder.encode(
    texts_only,
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False,   # IndexFlatL2 uses raw L2 — do NOT normalize here
).astype("float32")

# Shape and dtype validation
assert embeddings.ndim == 2, \
    f"Expected 2D array, got {embeddings.ndim}D"
assert embeddings.shape[0] == len(sub_chunks), \
    f"Row count mismatch: {embeddings.shape[0]} vs {len(sub_chunks)} sub-chunks"
assert embeddings.shape[1] == EMBEDDING_DIM, \
    f"Dimension mismatch: {embeddings.shape[1]} != {EMBEDDING_DIM}"
assert embeddings.dtype == np.float32, \
    f"Expected float32, got {embeddings.dtype}"
assert np.all(np.isfinite(embeddings)), \
    "Embeddings contain NaN or Inf values"

print(f"\nEmbeddings shape : {embeddings.shape}")
print(f"Dtype            : {embeddings.dtype}")
print(f"L2 norm of vec[0]: {np.linalg.norm(embeddings[0]):.4f}")

## Step 6 — Build FAISS Index and ID Map

In [ ]:
# ── Cell 7: FAISS IndexFlatL2 Build + id_map Construction ────────────────────
import faiss

# Build index — IndexFlatL2: exact L2 search, no training, 100% recall
index = faiss.IndexFlatL2(EMBEDDING_DIM)
index.add(embeddings)   # assigns IDs 0, 1, ..., N-1 in order

assert index.ntotal == len(sub_chunks), \
    f"FAISS vector count {index.ntotal} != sub-chunk count {len(sub_chunks)}"
assert index.ntotal > EXPECTED_UNIQUE_CHUNKS, (
    f"Re-chunking should produce > {EXPECTED_UNIQUE_CHUNKS} sub-chunks; "
    f"got {index.ntotal}. Check Cell 4."
)

print(f"FAISS index built: {index.ntotal} vectors, dim={index.d}")

# Build id_map — maps each FAISS integer ID (as string) to text + metadata
# String keys are required for JSON serialization
id_map = {
    str(i): {
        "text":               sub_chunks[i]["text"],
        "chunk_id":           f"chunk_{sub_chunks[i]['original_chunk_idx']:03d}",
        "original_chunk_idx": sub_chunks[i]["original_chunk_idx"],
    }
    for i in range(len(sub_chunks))
}

print(f"id_map entries: {len(id_map)}")
print(f"\nSample entry id='0':")
e = id_map["0"]
print(f"  chunk_id:           {e['chunk_id']}")
print(f"  original_chunk_idx: {e['original_chunk_idx']}")
print(f"  text (first 200):   {e['text'][:200]}")

## Step 7 — Persist Artifacts to Google Drive

`config.json` is written **last** and acts as a build-complete sentinel — if it is absent, the build did not finish.

In [ ]:
# ── Cell 8: Save Artifacts to Drive ──────────────────────────────────────────
import json
from datetime import date

# 1. FAISS index
faiss.write_index(index, INDEX_PATH)
print(f"FAISS index saved : {INDEX_PATH}")

# 2. ID map (ensure_ascii=False preserves special characters in Apple's report)
with open(IDMAP_PATH, "w", encoding="utf-8") as f:
    json.dump(id_map, f, ensure_ascii=False, indent=2)
print(f"id_map saved      : {IDMAP_PATH}")

# 3. Config — written last as a build-complete sentinel
config = {
    "encoder_model":        ENCODER_MODEL,
    "embedding_dim":        EMBEDDING_DIM,
    "chunk_size_tokens":    CHUNK_SIZE_TOKENS,    # 200 — actual value used for chunking
    "chunk_overlap_tokens": CHUNK_OVERLAP_TOKENS, # 20
    "length_function":      "token_count",         # token-aware splitter
    "max_tokens_limit":     MAX_TOKENS,
    "normalize_embeddings": False,
    "n_source_chunks":      EXPECTED_UNIQUE_CHUNKS,
    "n_sub_chunks":         index.ntotal,
    "hf_dataset":           HF_DATASET_ID,
    "faiss_index_type":     "IndexFlatL2",
    "index_metric":         "L2",
    "build_date":           date.today().isoformat(),
}
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)
print(f"config saved      : {CONFIG_PATH}")

print(f"\nArtifacts written to: {DRIVE_DIR}")

## Step 8 — Verification Suite

Ten checks that must all pass before this notebook is considered complete. Checks 2, 3, 6, and 10 depend on in-memory objects from earlier cells (`embeddings`, `encoder`, `token_lengths`); run this cell without restarting the kernel.

In [ ]:
# ── Cell 9: Verification Suite ────────────────────────────────────────────────
import os, json
import numpy as np
import faiss
import matplotlib.pyplot as plt

print("=" * 60)
print("VERIFICATION SUITE")
print("=" * 60)

# ── Check 1: Re-chunking produced more than 215 sub-chunks ───────────────────
print("\n[1] Re-chunking produced more than 215 sub-chunks...")
assert index.ntotal > EXPECTED_UNIQUE_CHUNKS, (
    f"FAIL: index.ntotal={index.ntotal} should be > {EXPECTED_UNIQUE_CHUNKS}. "
    f"Re-chunking may not have run."
)
print(f"    PASS: index.ntotal = {index.ntotal}")

# ── Check 2: All embedding vectors are finite ─────────────────────────────────
print("\n[2] All embedding vectors are finite (no NaN/Inf)...")
assert np.all(np.isfinite(embeddings)), (
    f"FAIL: embeddings contain {np.sum(~np.isfinite(embeddings))} non-finite values"
)
print(f"    PASS: {embeddings.shape[0]} vectors, all finite")

# ── Check 3: Embedding dimension == 384 ──────────────────────────────────────
print("\n[3] Embedding dimension == 384...")
assert embeddings.shape[1] == EMBEDDING_DIM, \
    f"FAIL: embeddings.shape[1]={embeddings.shape[1]}, expected {EMBEDDING_DIM}"
assert index.d == EMBEDDING_DIM, \
    f"FAIL: index.d={index.d}, expected {EMBEDDING_DIM}"
print(f"    PASS: dim = {embeddings.shape[1]}")

# ── Check 4: id_map keys are contiguous integers 0..N-1 ──────────────────────
print("\n[4] id_map keys are contiguous integers 0..N-1...")
expected_keys = set(str(i) for i in range(index.ntotal))
actual_keys   = set(id_map.keys())
assert expected_keys == actual_keys, (
    f"FAIL: id_map key mismatch. "
    f"Missing: {list(expected_keys - actual_keys)[:5]}. "
    f"Extra: {list(actual_keys - expected_keys)[:5]}."
)
print(f"    PASS: {len(id_map)} contiguous keys")

# ── Check 5: All id_map entries have required fields and valid values ─────────
print("\n[5] All id_map entries have required fields with valid values...")
required_fields = {"text", "chunk_id", "original_chunk_idx"}
for k, v in id_map.items():
    missing = required_fields - set(v.keys())
    assert not missing, f"FAIL: id_map['{k}'] missing fields: {missing}"
    assert isinstance(v["text"], str) and len(v["text"]) > 0, \
        f"FAIL: id_map['{k}']['text'] is empty or not a string"
    assert isinstance(v["original_chunk_idx"], int), \
        f"FAIL: id_map['{k}']['original_chunk_idx'] is not an int"
    assert 0 <= v["original_chunk_idx"] < EXPECTED_UNIQUE_CHUNKS, (
        f"FAIL: original_chunk_idx={v['original_chunk_idx']} out of [0, {EXPECTED_UNIQUE_CHUNKS})"
    )
print(f"    PASS: all {len(id_map)} entries valid")

# ── Check 6: Sample query returns topically relevant top-1 ───────────────────
print("\n[6] Sample query — 'Apple renewable energy percentage'...")
TEST_QUERY = "Apple renewable energy percentage"
query_vec  = encoder.encode([TEST_QUERY], convert_to_numpy=True).astype("float32")
distances, ids = index.search(query_vec, k=5)

relevant_keywords = {
    "renewable", "energy", "electricity", "clean", "solar", "wind",
    "carbon", "percent", "100%", "power", "grid", "emission"
}
top1_text      = id_map[str(ids[0][0])]["text"].lower()
top1_relevant  = any(kw in top1_text for kw in relevant_keywords)

print(f"    Top-5 results (L2 distances):")
for rank, (fid, dist) in enumerate(zip(ids[0], distances[0]), 1):
    text      = id_map[str(fid)]["text"]
    is_rel    = any(kw in text.lower() for kw in relevant_keywords)
    print(f"      [{rank}] id={fid}, dist={dist:.4f}, "
          f"relevant={'YES' if is_rel else 'NO'}, "
          f"preview='{text[:80]}'")

assert top1_relevant, (
    f"FAIL: Top-1 result for '{TEST_QUERY}' does not appear topically relevant. "
    f"Check embedding quality or re-chunking correctness."
)
print(f"    PASS: top-1 result contains relevant keywords")

# ── Check 7: All 3 artifact files exist on Drive ─────────────────────────────
print("\n[7] Artifact files exist on Drive...")
for path, label in [
    (INDEX_PATH,  "corpus_index.faiss"),
    (IDMAP_PATH,  "id_map.json"),
    (CONFIG_PATH, "config.json"),
]:
    assert os.path.isfile(path), f"FAIL: {label} not found at {path}"
    print(f"    PASS: {label} exists ({os.path.getsize(path)/1024:.1f} KB)")

# ── Check 8: corpus_index.faiss file size sanity ─────────────────────────────
print("\n[8] FAISS index file size sanity...")
faiss_bytes   = os.path.getsize(INDEX_PATH)
faiss_kb      = faiss_bytes / 1024
# Raw vector data floor: ntotal * dim * 4 bytes
expected_min_kb = (index.ntotal * EMBEDDING_DIM * 4) / 1024 * 0.8
assert faiss_bytes > 1000, \
    f"FAIL: corpus_index.faiss is only {faiss_bytes} bytes — likely corrupt"
assert faiss_kb >= expected_min_kb, (
    f"FAIL: file too small ({faiss_kb:.1f} KB), expected >= {expected_min_kb:.1f} KB"
)
print(f"    PASS: {faiss_kb:.1f} KB (minimum expected: {expected_min_kb:.0f} KB)")

# ── Check 9: Round-trip — reload index from Drive, verify same top-1 ─────────
print("\n[9] Round-trip: reload index from Drive, verify same top-1 result...")
index_rt = faiss.read_index(INDEX_PATH)
assert index_rt.ntotal == index.ntotal, (
    f"FAIL: reloaded index has {index_rt.ntotal} vectors, expected {index.ntotal}"
)
distances_rt, ids_rt = index_rt.search(query_vec, k=1)
assert ids_rt[0][0] == ids[0][0], (
    f"FAIL: round-trip top-1 mismatch. "
    f"Original: id={ids[0][0]}, reloaded: id={ids_rt[0][0]}"
)
print(f"    PASS: top-1 id={ids[0][0]} matches after reload from Drive")

# ── Check 10: 100% of sub-chunks <= 256 tokens (histogram) ───────────────────
print("\n[10] Token length histogram — 100% of sub-chunks within 256 tokens...")
pct_within = (token_lengths <= MAX_TOKENS).mean() * 100
assert pct_within == 100.0, (
    f"FAIL: {100.0 - pct_within:.2f}% of sub-chunks exceed {MAX_TOKENS} tokens"
)
plt.figure(figsize=(10, 4))
plt.hist(token_lengths, bins=40, edgecolor='black', color='steelblue', alpha=0.85)
plt.axvline(MAX_TOKENS, color='red', linestyle='--', linewidth=2,
            label=f'Hard limit ({MAX_TOKENS} tokens)')
plt.xlabel("Token count per sub-chunk (incl. special tokens)")
plt.ylabel("Count")
plt.title(f"Token Distribution — {len(sub_chunks)} sub-chunks ({pct_within:.1f}% within limit)")
plt.legend()
plt.tight_layout()
plt.show()
print(f"    PASS: 100% of {len(sub_chunks)} sub-chunks are within {MAX_TOKENS} tokens")

# ── Final Summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("ALL CHECKS PASSED")
print("=" * 60)
print(f"  Encoder model:   {ENCODER_MODEL}")
print(f"  Embedding dim:   {EMBEDDING_DIM}")
print(f"  Source chunks:   {EXPECTED_UNIQUE_CHUNKS}")
print(f"  Sub-chunks:      {index.ntotal}")
print(f"  Artifacts at:    {DRIVE_DIR}")
print(f"  Build date:      {config['build_date']}")
